# Tutorial 9 - Augment LLM with Retrieval Tool for Question-Answering

In this tutorial, we will use LangChain to create a typical RAG (Retrieval Augmented Generation) application for the Q&A task.

**LangChain** is an open-source package that aims to augment Large Language Models (LLMs) like GPT-3 with various tools to enhance their capabilities. It provides a framework for integrating LLMs with other tools such as search engines, databases, computational tools, and more.


**RAG** is a technique for augmenting LLM knowledge with additional data.
LLMs can reason about wide-ranging topics, but their knowledge is limited to the public data up to a specific point in time that they were trained on. If you want to build AI applications that can reason about private data or data introduced after a model’s cutoff date, you need to augment the knowledge of the model with the specific information it needs. The process of bringing the appropriate information and inserting it into the model prompt is known as Retrieval Augmented Generation (RAG).
LangChain has a number of components designed to help build Q&A applications, and RAG applications more generally.

In [1]:
! pip install langchain
! pip install sentence-transformers
! pip install langchain_community
! pip install langchain-huggingface
! pip install chromadb langchain-chroma
! pip install langchain-openai

#### Step 1: Indexing Load

To load private data into a RAG application, the initial step is to utilize DocumentLoaders. DocumentLoaders are specialized objects designed to retrieve and load data from a specific source. Here, we load our data from Google Drive.

A typical RAG application has two main components:


1.   Indexing: The indexing component of a RAG application involves a pipeline responsible for ingesting data from a given source and performing the necessary steps to prepare it for efficient retrieval. This process typically occurs offline, prior to the runtime of the application.


2.   Retrieval and generation: The retrieval and generation component forms the core of the RAG application. At runtime, when a user query is provided, this component is responsible for retrieving the relevant data from the previously indexed dataset. It leverages the indexing structure and algorithms to efficiently identify the most appropriate information for the given query. Once the relevant data has been retrieved, it is passed to the underlying model, which performs the generation step. The model uses the retrieved data, along with the query, to generate a coherent and contextually appropriate response. This response is then presented to the user as the output of the RAG application.


#### Step 2. Indexing: Split

The loaded document is often too long to fit in the context window of many models. Even for those models that could fit the full post in their context window, models can struggle to find information in very long inputs.

To handle this we’ll split the Document into chunks for embedding and vector storage. This should help us retrieve only the most relevant bits of the text at run time.

In this case we’ll split our documents into chunks of 1000 characters with 50 characters of overlap between chunks. The overlap helps mitigate the possibility of separating a statement from important context related to it. We use the RecursiveCharacterTextSplitter, which will recursively split the document using common separators like new lines until each chunk is the appropriate size. This is the recommended text splitter for generic text use cases.

We set add_start_index=True so that the character index at which each split Document starts within the initial Document is preserved as metadata attribute “start_index”.

In [2]:
from langchain_community.document_loaders import TextLoader
text_loader = TextLoader("data.txt")
document = text_loader.load()
document

[Document(metadata={'source': 'data.txt'}, page_content="Amit, a health-conscious man from suburban India, regularly visited his local doctor, Dr. Kapoor, for check-ups. After a routine check-up, Dr. Kapoor prescribed a medication to Amit for his recurring headaches. The doctor advised Amit to take the medicine only when needed, not exceeding a certain limit, as misuse could lead to side effects. Amit was concerned about how to keep track of his medication intake. That's when he learned about Healthify, a healthcare system that was highly accessible in his area.\n\nWith the help of Healthify, Amit's prescription and medication use became more manageable. The system allowed him to store his prescription digitally, complete with reminders on when to take the medicine. Healthify also kept track of the number of pills he had taken, making it impossible for him to exceed the recommended limit.\n\nHealthify not only helped Amit but also indirectly contributed to improving healthcare on a bro

In [3]:
def split_text_into_lines(text, width=110):
    lines = text.split("\n")
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]
    wrapped_text = "\n".join(wrapped_lines)
    return wrapped_text

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=50, add_start_index=True
)

document_chunks = text_splitter.split_documents(document)

In [5]:
len(document_chunks)

3

#### Step 3. Indexing: Store

Now we need to index our document chunks so that we can search over them at runtime. The most common way to do this is to embed the contents of each document split and insert these embeddings into a vector database (or vector store). When we want to search over our splits, we take a text search query, embed it, and perform some sort of “similarity” search to identify the stored splits with the most similar embeddings to our query embedding.

We can embed and store all of our document splits in a single command using the chromadb vector store and HuggingFaceEmbeddings model.

The HuggingFaceEmbeddings component acts as a wrapper around a text embedding model, which is responsible for converting textual input into dense vector representations, also known as embeddings.

The VectorStore component serves as a wrapper around a vector database, specifically designed for storing and querying embeddings. It provides an interface to interact with the underlying database that efficiently manages the storage and retrieval of vectors. In the case of a RAG application, the chromadb VectorStore is commonly used.



In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB is a vector database designed for managing and querying high-dimensional vectors, which are commonly used in machine learning and artificial intelligence applications, particularly in areas like natural language processing. It provides efficient storage, indexing, and retrieval capabilities for vector data, enabling fast similarity searches and nearest neighbor queries. ChromaDB is optimized for handling large-scale datasets and supports real-time operations, making it suitable for use cases such as recommendation systems, semantic search, and clustering. Its architecture is designed to integrate seamlessly with modern AI workflows, offering scalability and flexibility for developers and researchers working with complex data.

In [7]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=document_chunks,
    embedding=embeddings
)

#### Step 4. Retrieve Answers from Documents

Now let’s write the actual application logic. We want to create a simple application that takes a user question, and searches for documents relevant to that question.

We use the ChatOpenAI function in the LangChain library, which is used to load a chain that enables question-answering functionality over a set of documents.

We use DeepSeek-R1 model as the fundation QA model. The DeepSeek-R1 is a state-of-the-art foundational language model. It is designed to assist researchers in advancing their work in the field of artificial intelligence.  These models are trained on a diverse range of internet text, enabling them to generate coherent and contextually relevant text based on the input they receive. DeepSeek-R1 is intended to be more efficient and accessible, providing a powerful tool for natural language processing tasks while being optimized for performance on lower-resource hardware compared to some other large language models.

In [8]:
import os
from langchain_huggingface import HuggingFaceEndpoint

from langchain_openai import ChatOpenAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
chat_model = ChatOpenAI(api_key="HF_TOKEN", # 建议改成 HF_TOKEN
              base_url="https://router.huggingface.co/v1",
              model="deepseek-ai/DeepSeek-R1-0528:fastest",
              temperature=0,
              max_tokens=512)

prompt = ChatPromptTemplate.from_messages([
    ("system", """Use the following context to answer the question. If you don't know, say you don't know.
Context: {context}"""), ("human", "{input}") ])


Here we define our question

In [9]:
question = "What is the name of doctor & patient ?"

We use the vector database ChromaDB to search for the text chunks most related to the question, in order to assist the LLM in answering the question.

In [10]:
search_results = vector_store.similarity_search(question)
search_content = search_results[0].page_content
print(search_content)

Amit, a health-conscious man from suburban India, regularly visited his local doctor, Dr. Kapoor, for check-ups. After a routine check-up, Dr. Kapoor prescribed a medication to Amit for his recurring headaches. The doctor advised Amit to take the medicine only when needed, not exceeding a certain limit, as misuse could lead to side effects. Amit was concerned about how to keep track of his medication intake. That's when he learned about Healthify, a healthcare system that was highly accessible in his area.

With the help of Healthify, Amit's prescription and medication use became more manageable. The system allowed him to store his prescription digitally, complete with reminders on when to take the medicine. Healthify also kept track of the number of pills he had taken, making it impossible for him to exceed the recommended limit.


In [11]:
qa_chain = create_stuff_documents_chain(chat_model, prompt)
answer = qa_chain.invoke({ "context": search_results, "input": question, })

print(answer)

Based solely on the provided context:

1.  **Patient:** Amit
2.  **Doctor:** Dr. Kapoor

These names are explicitly stated in the first paragraph of the context.
